## Importar librerías e instalar paquetes

In [7]:
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoModelForImageClassification, AutoFeatureExtractor

from torchvision import transforms
from tqdm import tqdm
import sklearn.metrics as metrics
import pandas as pd

## Usar GPU

In [8]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", DEVICE)

Usando dispositivo: cuda


## Características de cada modelo

In [9]:
model_data = {
    "swinv2": {
        "image_size": 256,
        "model_name": "mirluvams/swinv2-base-patch4-window16-256-popocatepetl"
    }
}

In [10]:
#model_data["swinv2"]["model_name"]

## Cargar dataset de prueba

In [11]:
dataset = load_dataset("mirluvams/popocatepetl")
test_dataset = dataset["test"]

Using the latest cached version of the dataset since mirluvams/popocatepetl couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\alexi\.cache\huggingface\datasets\mirluvams___popocatepetl\default\0.0.0\df77c35c9e03cda387c6726f64fd77dd64c85591 (last modified on Sun Oct 12 03:15:20 2025).


In [12]:
print("Columnas:", test_dataset.column_names)
print("Cant. de imgs de test:", len(test_dataset))

Columnas: ['label', 'image', 'video', 'id']
Cant. de imgs de test: 24040


In [13]:
dataset["test"].set_format("torch")

In [14]:
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'image', 'video', 'id'],
        num_rows: 216352
    })
    test: Dataset({
        features: ['label', 'image', 'video', 'id'],
        num_rows: 24040
    })
})

## Función de evaluación de los modelos

In [19]:
BATCH_SIZE=16
def evaluate_model(model_name, image_size, **kwargs):
    print(f"\nEvaluando modelo: {model_name}")
    
    # Cargar modelo
    model = AutoModelForImageClassification.from_pretrained(model_name).to(DEVICE)
    
    # Transformar datos
    _test_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                             std=[0.229, 0.224, 0.225]),
    ])
    
    def test_transform(ex):
        ex["pixel_values"] = [_test_transform(image) for image in ex["image"]]
        return ex
   
    dataset["test"].set_transform(test_transform)

    # Evaluar
    preds, labels = [], []
    model.eval()
    with torch.no_grad():
        for batch in tqdm(dataset["test"].iter(BATCH_SIZE), desc="Evaluando", total=len(dataset["test"])//BATCH_SIZE+1):
            pixel_values = torch.stack(batch["pixel_values"]).to(DEVICE)
            batch_labels = batch["label"]
            
            outputs = model(pixel_values=pixel_values)
            logits = outputs.logits
            predictions = torch.argmax(logits, dim=-1)
            
            preds.extend(predictions.cpu().numpy())
            labels.extend(batch_labels)
    
    # Calcular metricas
    acc = metrics.accuracy_score(labels, preds)
    f1 = metrics.f1_score(labels, preds, average="weighted")
    f1_micro = metrics.f1_score(labels, preds, average="micro")
    f1_macro = metrics.f1_score(labels, preds, average="macro")
    
    print(f"Accuracy: {acc:.4f} | F1: {f1:.4f} | F1_micro: {f1_micro:.4f} | F1_macro: {f1_macro:.4f}")

    # Matriz de confusion y reporte
    class_names = dataset["test"].features["label"].names
    report_dict = metrics.classification_report(
     labels, preds,
     target_names=class_names,
     digits=4,
     output_dict=True,
     zero_division=0,
    )
    cm = metrics.confusion_matrix(labels, preds, labels=list(range(len(class_names))))
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    model_short_name = model_name.split("/")[-1]
    pd.DataFrame(report_dict).T.to_csv(f"{model_short_name}_report.csv")
    cm_df.to_csv(f"{model_short_name}_confusion.csv")
    
    return {"model": model_name, "accuracy": acc, "f1": f1, "f1_micro": f1_micro, "f1_macro": f1_macro}

## Resultados de evaluación de los modelos

In [20]:
#evaluate_model(model_data["swinv2"]["model_name"], model_data["swinv2"]["image_size"])
#for model in model_data.keys():
#   evaluate_model(**model_data["swinv2"])

In [21]:
results = []

In [22]:
for model in model_data.keys():
    result = evaluate_model(**model_data[model])
    results.append(result)


Evaluando modelo: mirluvams/swinv2-base-patch4-window16-256-popocatepetl


Evaluando: 100%|███████████████████████████████████████████████████████████████████| 1503/1503 [01:48<00:00, 13.82it/s]

Accuracy: 0.9894 | F1: 0.9894 | F1_micro: 0.9894 | F1_macro: 0.9864


In [23]:
df_results_finetuned = pd.DataFrame(results)

In [24]:
df_results_finetuned

,model,accuracy,f1,f1_micro,f1_macro
0,mirluvams/swinv2-base-patch4-window16-256-popo...,0.989434,0.98944,0.989434,0.986394


In [25]:
df_results_finetuned.to_csv('results_finetuned.csv')